# 02 Company profiles (facets)

**What this does.** Describes each company's business model in the same fields, so companies can be compared.
- **General facets (current):** 12 facets (F1 offering … F12 regulatory exposure) filled by Claude Sonnet 5.5 through
  the Claude Code CLI (Matthew and Richard, 2026-11-04), prompt `2026-10-25k`. First pass: company name, website and
  model knowledge. A second pass reads the saved website pages only for facets still unknown; a third checks
  multi-business splits. The model never sees SIC codes, tickers or financial fields. Company size (F9) is filled but
  left out of B3.
- **Sector facets (Fiona):** sector-specific fields; her sub-sector feeds the current B3 variant.
- **Industry facets (Barbara):** SIC2 modules from her dictionary; not in B3.
- **Divestiture corrections (Yiming):** a copy of the general facets with activities the companies have exited
  removed (42 companies, flagged `divestiture_corrected`; built by v1's `integration_sources.py`). Kept, not the
  default: Barbara's 293 pairs and the current B3 list use the uncorrected fills.

**Saved outputs:** `outputs-local/v1a/02_profiles/` (tables). The raw model answers stay in `outputs-local/v1/`
(linked in `seed_manifest.json`).

**Changelog**
- 2026-11-18: created from `v1/notebooks/comp_universe_and_facets.ipynb`, `facet_fill_runs.ipynb`,
  `business_structure_eval.ipynb` and `business_split_refinement_eval.ipynb`. The three tables rebuild exactly
  from the saved raw answers.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v1a" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v1a" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

## 1. Current setting (`v1a/current.toml`)

In [2]:
pd.Series(cur["profiles"])

general                                          outputs-local/v1a/02_profiles/facets_sonnet/tables/facets.csv
general_prompt_version                                                                             2026-10-25k
sector                             outputs-local/v1a/02_profiles/sector_facets_sonnet/tables/sector_facets.csv
industry                  outputs-local/v1a/02_profiles/industry_facets_sonnet/tables/industry_facets_long.csv
corrected_general                                    outputs-local/v1a/02_profiles/facets_sonnet_corrected.csv
website_pages                                                           outputs-local/website_scrape/page_text
dtype: object

## 2. Inputs

In [3]:
nb.inputs([cur["profiles"][k] for k in ("general", "sector", "industry", "corrected_general")])

,input,rows,sha256,status
0,outputs-local/v1a/02_profiles/facets_sonnet/tables/facets.csv,533,8619ecf21c32,matches seed
1,outputs-local/v1a/02_profiles/sector_facets_sonnet/tables/sector_facets.csv,533,a7139d71efd6,matches seed
2,outputs-local/v1a/02_profiles/industry_facets_sonnet/tables/industry_facets_long.csv,5331,8b7b986d8697,matches seed
3,outputs-local/v1a/02_profiles/facets_sonnet_corrected.csv,533,ba9a51e7b323,matches seed


## 3. Results

### 3.1 General facets: what was filled

In [4]:
import facets as fc, facet_weighting as fw
general = pd.read_csv(paths.path(cur["profiles"]["general"]), dtype=str).fillna("")
names = list(fc.FACET_ID)
used = set(fw.similarity_facets())
status = pd.DataFrame({n: general[f"{n}_status"].value_counts() for n in names}).T.fillna(0).astype(int)
status.insert(0, "id", [fc.FACET_ID[n] for n in names])
status["in B3"] = ["yes" if n in used else "no" for n in names]
print(f"{general['IQ_COMPANY_ID'].nunique()} companies, {len(general)} business rows; prompt {general['prompt_version'].unique().tolist()}; "
      f"status {general['llm_status'].value_counts().to_dict()}")
status

533 companies, 533 business rows; prompt ['2026-10-25k']; status {'ok': 533}


,id,filled,not_applicable,in B3
offering_description,F1,533,0,yes
primary_products,F2,419,114,yes
primary_services,F3,414,119,yes
customer_segment,F4,533,0,yes
end_markets,F5,533,0,yes
revenue_model,F6,533,0,yes
value_chain_position,F7,533,0,yes
geography_served,F8,533,0,yes
company_size,F9,533,0,no
specialization,F10,533,0,yes


### 3.2 Where the values come from

Each facet value records its source: model knowledge, or the website lines it cites. The website pass only runs
for facets the first pass left unknown.

In [5]:
src = pd.DataFrame({n: general[f"{n}_source"] for n in names})
kind = src.apply(lambda col: col.map(lambda s: "website" if "http" in s else ("model knowledge" if "model knowledge" in s else "none")))
by_source = kind.apply(lambda c: c.value_counts()).T.fillna(0).astype(int)
display(by_source)
print("share of filled values from model knowledge only:",
      f"{(kind == 'model knowledge').sum().sum() / (kind != 'none').sum().sum():.1%}")
print("companies with at least one website-sourced value:", int((kind == "website").any(axis=1).sum()))

,model knowledge,none,website
offering_description,523,0,10
primary_products,526,3,4
primary_services,523,0,10
customer_segment,523,0,10
end_markets,525,0,8
revenue_model,524,1,8
value_chain_position,525,0,8
geography_served,525,0,8
company_size,527,2,4
specialization,524,0,9


share of filled values from model knowledge only: 98.5%
companies with at least one website-sourced value: 10


### 3.3 Sector facets (Fiona)

In [6]:
sector = pd.read_csv(paths.path(cur["profiles"]["sector"]), dtype=str).fillna("")
print(f"{len(sector)} companies; status {sector['llm_status'].value_counts().to_dict()}; prompt {sector['prompt_version'].unique().tolist()}")
print("sub-sector (used by B3):", sector["sub_sector_status"].value_counts().to_dict())
sector[["IQ_COMPANY_ID", "sector", "sub_sector"]].head(8)

533 companies; status {'ok': 533}; prompt ['2026-10-27b']
sub-sector (used by B3): {'stated': 533}


,IQ_COMPANY_ID,sector,sub_sector
0,IQ100918,energy_resources,"integrated electric and gas utility, Pacific Northwest"
1,IQ100935,construction_real_estate,"commercial mechanical, electrical and plumbing contracting; building service and maint..."
2,IQ1032802,software,investor communications and capital markets post-trade fintech; wealth management tech...
3,IQ105613224,manufacturing,Industrial electric heat tracing and process heating; industrial boilers
4,IQ1064486,manufacturing,payment card manufacturing; instant issuance and card fulfillment services; prepaid ca...
5,IQ1067110,business_services,digital learning courseware and education publishing; medical education
6,IQ107405,construction_real_estate,single-family homebuilding; mortgage and title services; build-to-rent homes
7,IQ108424,distribution_transport,industrial MRO supplies distribution; metalworking tools and supplies


### 3.4 Industry facets (Barbara's SIC2 modules)

In [7]:
industry = pd.read_csv(paths.path(cur["profiles"]["industry"]), dtype=str)
share = industry["status"].value_counts(normalize=True).mul(100).round(1)
print(f"{len(industry)} facet rows for {industry['IQ_COMPANY_ID'].nunique()} companies; status %:", share.to_dict())
industry.groupby("industry")["status"].value_counts(normalize=True).unstack().fillna(0).mul(100).round(0).head(12)

5331 facet rows for 533 companies; status %: {'stated': 48.2, 'not_applicable': 28.6, 'unknown': 23.2}


status,not_applicable,stated,unknown
industry,,,
Agricultural Services,0.0,80.0,20.0
Amusement & Recreation,27.0,45.0,28.0
Apparel & Finished Textile Products,3.0,87.0,10.0
Building Construction Contractors and Operative Builders,3.0,68.0,29.0
Business Services,45.0,34.0,20.0
Chemicals & Allied Products,19.0,62.0,19.0
Communications,20.0,54.0,27.0
Depository Institutions,100.0,0.0,0.0
Eating & Drinking Places,0.0,42.0,58.0


### 3.5 Divestiture corrections (Yiming; kept, not the default)

In [8]:
corrected = pd.read_csv(paths.path(cur["profiles"]["corrected_general"]), dtype=str).fillna("")
key = ["IQ_COMPANY_ID", "business_id"]
a, b = general.set_index(key).sort_index(), corrected.set_index(key).sort_index()[general.columns.drop(key)]
diff = (a != b)
changes = [(i[0], c, a.at[i, c][:70], b.at[i, c][:70]) for i, row in diff.iterrows() for c in row.index[row]]
changes = pd.DataFrame(changes, columns=["IQ_COMPANY_ID", "column", "current (Sonnet)", "corrected"])
universe = pd.read_csv(paths.path(cur["universe"]["universe"]), dtype=str).set_index("IQ_COMPANY_ID")
changes.insert(1, "ticker", changes["IQ_COMPANY_ID"].map(universe["target_ciq_ticker"]))
value_changes = changes[~changes["column"].str.endswith(("_source", "_flags", "_status", "_category"))]
print(f"{value_changes['IQ_COMPANY_ID'].nunique()} companies, {len(value_changes)} value fields changed; by facet:",
      value_changes["column"].value_counts().to_dict())
value_changes.head(12)

42 companies, 116 value fields changed; by facet: {'offering_description': 28, 'specialization': 27, 'primary_products': 24, 'end_markets': 22, 'primary_services': 15}


,IQ_COMPANY_ID,ticker,column,current (Sonnet),corrected
0,IQ121509,ZD,offering_description,Operates a portfolio of digital media brands and online properties cov,Operates a portfolio of digital media brands and online properties cov
1,IQ121509,ZD,primary_products,digital media websites; VPN and cybersecurity software; email marketin,digital media websites; VPN and cybersecurity software; email marketin
2,IQ121509,ZD,primary_services,digital advertising; lead generation; performance marketing; email and,digital advertising; lead generation; performance marketing; email and
3,IQ121509,ZD,specialization,"related lines: digital media, performance marketing, cybersecurity and","related lines: digital media, performance marketing, cybersecurity and"
4,IQ130863,FSTR,offering_description,Manufactures and supplies engineered products for rail and transit inf,Manufactures and supplies engineered products for rail and transit inf
5,IQ130863,FSTR,primary_products,rail track components and fasteners; friction management systems; prec,friction management systems; precast concrete products; protective coa
6,IQ168154,AIR,primary_products,aircraft parts and components; engineered pallets and shelters for mil,aircraft parts and components; engineered pallets and shelters for mil
7,IQ170928919,TRU,end_markets,financial services; insurance; healthcare; telecommunications; retail;,financial services; insurance; telecommunications; retail; government
8,IQ18671,ALB,offering_description,"Produces lithium compounds for batteries, bromine-based flame retardan","Produces lithium compounds for batteries, bromine-based flame retardan"
9,IQ18671,ALB,primary_products,lithium carbonate and hydroxide; bromine and brominated flame retardan,lithium carbonate and hydroxide; bromine and brominated flame retardan


## 4. Checks against the saved v1 numbers

In [9]:
import contextlib, io, shutil, tempfile
import fill_facets as ff, fill_sector_facets as fs, fill_industry_facets as fi
rebuild = [("general", "facets_sonnet", lambda d: ff.write_tables(d), "tables/facets.csv", cur["profiles"]["general"]),
           ("sector", "sector_facets_sonnet", lambda d: fs.write_tables(fs.output_paths(d)), "tables/sector_facets.csv", cur["profiles"]["sector"]),
           ("industry", "industry_facets_sonnet", lambda d: fi.write_tables(fi.output_paths(d)), "tables/industry_facets_long.csv", cur["profiles"]["industry"])]
with tempfile.TemporaryDirectory() as tmp:               # rebuild from the raw answers in the v1 record (read only)
    for name, folder, write, table, seeded in rebuild:
        d = Path(tmp) / folder
        shutil.copytree(paths.V1_OUTPUTS / folder / "profiles", d / "profiles")
        with contextlib.redirect_stdout(io.StringIO()):    # the writers print temp paths
            write(d)
        x, y = pd.read_csv(d / table, dtype=str).fillna(""), pd.read_csv(paths.path(seeded), dtype=str).fillna("")
        checks.add(f"{name} table rebuilds exactly from the raw answers", list(x.columns) == list(y.columns) and x.equals(y), f"{x.shape}")
checks.add("general: 533 companies, all ok, prompt 2026-10-25k", general["IQ_COMPANY_ID"].nunique() == 533
           and general["llm_status"].eq("ok").all() and general["prompt_version"].eq(cur["profiles"]["general_prompt_version"]).all())
checks.add("sector: 533 companies, sub-sector stated for all", len(sector) == 533 and sector["sub_sector_status"].eq("stated").all())
checks.add("industry: 5,331 rows (v1 handoff 2026-11-04)", len(industry) == 5331, str(share.to_dict()))
flagged = set(corrected.loc[corrected["divestiture_corrected"] == "True", "IQ_COMPANY_ID"])
checks.add("corrections are exactly the 42 flagged companies", set(value_changes["IQ_COMPANY_ID"]) == flagged and len(flagged) == 42,
           f"{len(value_changes)} value fields")
checks.show()

7/7 checks pass


,check,result,detail
0,general table rebuilds exactly from the raw answers,pass,"(533, 74)"
1,sector table rebuilds exactly from the raw answers,pass,"(533, 199)"
2,industry table rebuilds exactly from the raw answers,pass,"(5331, 12)"
3,"general: 533 companies, all ok, prompt 2026-10-25k",pass,
4,"sector: 533 companies, sub-sector stated for all",pass,
5,"industry: 5,331 rows (v1 handoff 2026-11-04)",pass,"{'stated': 48.2, 'not_applicable': 28.6, 'unknown': 23.2}"
6,corrections are exactly the 42 flagged companies,pass,116 value fields


## 5. History (v1)

- **Haiku fills** (`outputs-local/v1/facets/`, 2026-10-25) were the v0/v1 profiles. Sonnet replaced them on
  2026-11-04; Yifan's check found Haiku still described businesses CSL, AFG and AZZ had sold.
- **Business split pilot** (`business_split.py`, `business_split_refinement.py`): source-checked business groups
  for 40 companies; kept as metadata, not adopted as the peer rule (v1 handoff 2026-10-28).
- **Facet verification** (`scripts/verification/`): evidence retrieval and checks of facet claims (Yiming).
- **Alternatives tried in the 11-10 integration study** (stay in v1): GPT-6.1 Sol fills, cited-only fills,
  Yifan's Orbis-aligned fields. No measured gain over these profiles; see `09_analysis`.